# DAY 2 — 바이브 코딩 기반 현장 데이터 분석

**AI 적용을 위한 현장 데이터 수집 및 디지털화** · DAY 2 실습

교재 PART 02 (pp.62–126) 의 실습 코드를 코랩에서 바로 돌릴 수 있게 옮긴 것입니다.
설명과 그림은 교재와 학습사이트에 있습니다 — https://build-data.jobability.co.kr

---
### 코랩 쓰는 법 세 가지만

1. **셀 실행** — 코드 칸 왼쪽 ▶ 를 누르거나 `Shift + Enter`.
2. **위에서부터 차례로** — 앞 칸에서 만든 것을 뒤 칸이 씁니다. 건너뛰면 오류가 납니다.
3. **내 사본으로** — 「파일 → 드라이브에 사본 저장」을 먼저 해야 고친 내용이 남습니다.

> **제목의 번호는 교재 절 번호입니다.** 이론만 있는 절은 실행할 것이 없어 빠져 있어서
> 번호가 건너뜁니다(0 → 4 처럼). 빠진 절의 설명은 교재와 학습사이트에 있습니다.

> **실습 데이터는 준비 칸이 자동으로 받아 옵니다.** 교재와 같은 원본이라 설명 글의 숫자가
> 그대로 나옵니다. 내려받기가 막혀 대체 데이터로 진행한 경우에는 행 수·건수처럼 정해 둔 값은
> 같지만 평균·p값처럼 난수에 걸리는 값이 달라집니다 — **읽는 방법이 같으면 맞게 하신 것입니다.**

> **API 키는 왼쪽 🔑(보안 비밀)에 한 번 등록해 두면 편합니다.**
> 이름은 `OPENAI_API_KEY`, 값은 강사가 나눠 준 키(`sk-…`), 그리고 **「노트북 액세스」를 켜면**
> 아래 키 칸이 묻지 않고 그냥 지나갑니다.
> 이 키는 **실습용 임시 공용 키**로 수업이 끝나면 폐기됩니다 — 공유하지 마세요.

## 0. 실습 준비

이 아래 두 칸은 세션을 새로 열 때마다 한 번씩 실행합니다.

In [ ]:
# 그래프에 한글이 네모로 나오지 않게 폰트를 깝니다. 세션마다 한 번만 하면 됩니다.
!apt-get -qq install fonts-nanum > /dev/null 2>&1
import matplotlib, matplotlib.pyplot as plt
from matplotlib import font_manager
import glob, os
cand = glob.glob("/usr/share/fonts/truetype/nanum/NanumGothic*.ttf") + glob.glob("fonts/NanumGothic*.ttf")
if not cand:                      # apt 가 막힌 환경이면 자료 저장소에서 받아 씁니다
    import urllib.request
    os.makedirs("fonts", exist_ok=True)
    urllib.request.urlretrieve(
        "https://raw.githubusercontent.com/aebonlee/materials/main/build-data/data/NanumGothic-Regular.ttf",
        "fonts/NanumGothic-Regular.ttf")
    cand = glob.glob("fonts/NanumGothic*.ttf")
if cand:
    font_manager.fontManager.addfont(cand[0])
    plt.rcParams["font.family"] = font_manager.FontProperties(fname=cand[0]).get_name()
plt.rcParams["axes.unicode_minus"] = False
print("한글 폰트:", plt.rcParams["font.family"])

In [ ]:
# 실습 데이터 준비
# 1) 왼쪽 폴더 아이콘에 data_day2.zip 을 올려 두었으면 그것을 씁니다.
# 2) 없으면 자료 저장소에서 자동으로 내려받습니다.
# 3) 그것도 막히면 같은 구조의 대체 데이터를 그 자리에서 만들어 씁니다.
import os, zipfile, urllib.request
if not os.path.exists("records"):
    if not os.path.exists("data_day2.zip"):
        try:
            print("data_day2.zip 을 자료 저장소에서 받는 중 …")
            urllib.request.urlretrieve("https://raw.githubusercontent.com/aebonlee/materials/main/build-data/data_day2.zip", "data_day2.zip")
        except Exception as e:
            print("  내려받지 못했습니다:", type(e).__name__)
    if os.path.exists("data_day2.zip"):
        zipfile.ZipFile("data_day2.zip").extractall(".")
    else:
        print("대신 같은 구조의 대체 실습 데이터를 만듭니다. 30초쯤 걸립니다.")
        import sys, subprocess
        urllib.request.urlretrieve("https://raw.githubusercontent.com/aebonlee/materials/main/build-data/data/make_day2_data.py", "make_day2_data.py")
        subprocess.run([sys.executable, "make_day2_data.py"], check=True)
print("준비된 폴더:", sorted(d for d in os.listdir(".") if os.path.isdir(d) and not d.startswith(".")))

## 1. 바이브 코딩의 원리

### 자연어 기반 코드 생성의 원리

**[2-1] 같은 작업을 지시하는 두 가지 방법**

```
[막연한 지시]
정비 데이터 좀 정리해 줘.
[구체적인 지시]
정비이력 CSV를 읽어서 행수, 컬럼, 앞 5행을 보여 줘.
각 컬럼의 자료형과 결측 개수도 정리해 줘.
```

## 4. 실습: 정비 이력 구조 확인

### 실습 준비: Colab 데이터 업로드

**[2-3] 데이터 압축 풀기**

화면은 2026년 7월 기준 Google Colab 화면입니다. 셀 왼쪽의 실행 버튼을 클릭하면 코드가 실행되며, 결과는 셀 아래에 표시됩니다. 먼저 압축을 풀겠습니다. 아래 코드는 압축이 이미 풀려 있으면 그냥 지나가고, 아니면 풀어 줍니다.

In [ ]:
import os, zipfile
if not os.path.exists("records"):          # 아직 압축을 풀지 않았다면
    if os.path.exists("data_day2.zip"):
        zipfile.ZipFile("data_day2.zip").extractall(".")
    else:
        raise SystemExit("데이터가 없습니다 — 위쪽 「실습 데이터 준비」 칸을 먼저 실행하세요.")
print("데이터 준비 상태:", sorted(d for d in os.listdir(".") if os.path.isdir(d) and not
d.startswith(".")))

실행 결과는 다음과 같습니다. 데이터 준비 상태데이터 준비 상태: ['fonts', 'kpi', 'parts', 'records']폴더 네 개가 만들어졌습니다. records에 정비 이력이, kpi에 장비 가동 기록이, parts에 부품 파일들이 들어 있고, fonts에는 그래프에 한글을 쓰기 위한 글꼴 파일이 있습니다. 이 네 개가 보이지 않으면 압축 파일이 제대로 올라가지 않은 것이니 업로드부터 다시 확인하기 바랍니다. 다음 코드를 실행하여 업로드한 데이터의 압축을 풉니다.

**[2-5] 분석 환경과 한글 폰트 설정**

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import font_manager
# 그래프 한글 폰트(나눔고딕, 동봉 파일)
import glob
_font = (glob.glob("fonts/NanumGothic*.ttf")
         + glob.glob("/usr/share/fonts/truetype/nanum/NanumGothic*.ttf"))
if _font:
    font_manager.fontManager.addfont(_font[0])
    plt.rcParams["font.family"] = font_manager.FontProperties(fname=_font[0]).get_name()
else:
    print("한글 폰트를 찾지 못했습니다 — 맨 위 「한글 폰트」 칸을 먼저 실행하세요.")
plt.rcParams["axes.unicode_minus"] = False
print("pandas", pd.__version__, "| 폰트 설정 완료")

pandas는 표 형태의 데이터를 다루는 도구이고, matplotlib은 그래프를 그리는 도구입니다. 지금은 이름만 알아 두면 됩니다. 뒤의 세 줄은 그래프에 한글을 쓰기 위한 설정입니다. 배포 번들에 글꼴 파일을 함께 넣어 두었기 때문에 이 코드는 Colab에서든 개인 PC에서든 똑같이 동작합니다. 이 설정을 빠뜨리면 나중에 그래프의 한글이 네모 상자로 나옵니다. 버전 번호는 실행하는 시점에 따라 다르게 표시될 수 있습니다. 오류 없이 "폰트 설정 완료"가 나오면 준비가 끝난 것입니다.

### 실습 시범: 데이터 기본 구조 확인

**[2-8] 정비 이력 파일 읽기**

정비 이력 파일을 불러와 데이터 구조를 확인하기 위해 다음과 같이 지시합니다. 정비 이력 구조 확인정비이력 CSV를 읽어서 행수, 컬럼, 앞 5행을 보여줘. 각 컬럼의 자료형과 결측 개수도 정리해줘. 받은 코드는 다음과 같습니다.

In [ ]:
df = pd.read_csv("records/maintenance_2024_2025.csv")
print(df.shape)
df.head()

실행 결과는 다음과 같습니다. 정비 이력 앞 5행(500, 9)record_id 장비ID 고장일시 구분 증상 원인부품 \0 MR-00001 EX-203 2024-01-04 12:10 정기점검 200시간 정기점검 엔진오일 필터1 MR-00002 EX-303 2024-01-04 16:10 고장수리 시동 불량(예열 지연) 연료 필터 카트리지2 MR-00003 EX-103 2024-01-06 08:10 정기점검 200시간 정기점검 엔진오일 필터3 MR-00004 EX-201 2024-01-07 07:40 고장수리 버킷 투스 파손 버킷 투스4 MR-00005 EX-304 2024-01-07 09:20 고장수리 배터리 방전으로 시동 불가 배터리조치방법 정비시간(분) 정비기사0 엔진오일·오일필터 교환, 벨트 장력 점검 40.0 유 기사1 예열 플러그 점검, 연료 계통 수분 배출 후 재시동 295.0 오 기사2 엔진오일·오일필터 교환, 벨트 장력 점검 90.0 박 기사3 파손 투스 교체, 고정 핀 상태 확인 120.0 정 기사4 배터리 교체 후 충전 계통 점검 80.0 오 기사첫 줄의 (500, 9)는 500행 9열이라는 뜻입니다. 앞 5행을 보면 컬럼 아홉 개가 무엇을 담는지 짐작할 수 있습니다.

**[2-10] 컬럼 자료형 확인**

다음은 각 컬럼이 어떤 자료형으로 읽혔는지 확인할 차례입니다.

In [ ]:
df.info()

두 가지를 읽어야 합니다. 먼저 자료형입니다. 정비시간(분)만 실수형(float64)이고 나머지는 전부 문자형(str)입니다. 정비시간은 40, 90, 120처럼 정수로 적혀 있는데 왜 실수로 읽혔을까요. 비어 있는 칸이 섞여 있으면 pandas가 그 열을 실수형으로 처리하기 때문입니다. 자료형이 예상과 다르면 그 열에 무언가 있다는 신호입니다. 또 하나 눈여겨볼 것은 고장일시가 문자형으로 읽혔다는 점입니다. 사람 눈에는 날짜지만 지금은 그냥 글자입니다. 이대로는 시간 계산을 할 수 없어서, 뒤의 전처리 실습에서 날짜형으로 바꾸는 작업을 하게 됩니다. 다음은 값이 채워진 개수입니다. 대부분 500인데 원인부품이 455, 정비시간(분)이 480입니다. 비어 있는 자리를 직접 세어 보겠습니다.

**[2-12] 컬럼별 결측 세기**

In [ ]:
df.isna().sum()

정비시간이 20건 비어 있습니다. 전체 500건의 4%입니다. 원인부품은 45건이 비어 있는데, 이쪽은 성격이 다릅니다. 정기점검처럼 특정 부품의 고장이 아닌 경우에는 원인부품을 적을 것이 없기 때문입니다. 같은 결측이라도 "적어야 하는데 빠진 것"과 "적을 것이 없는 것"은 구분해서 다뤄야 합니다. 정비시간 20건은 앞쪽에 해당합니다. 수리를 했으면 시간이 있었을 텐데 기록되지 않은 것입니다. 이 20건을 지울지, 다른 기록으로 복원할지는 다음 실습의 판단 문제입니다. 지금은 20건이라는 사실만 확인해 둡니다.

**[2-14] 정비 구분별 건수**

이제 값의 분포를 보겠습니다. 먼저 구분입니다.

In [ ]:
df["구분"].value_counts()

고장수리 270건, 정기점검 230건입니다. 절반 가까이가 정기점검이라는 사실이 중요합니다. "정비 건수가 많다"는 말을 할 때 정기점검까지 세면 안 되기 때문입니다. 정기점검은 시간이 되면 하는 것이지 장비에 문제가 있어서 하는 것이 아닙니다. 뒤의 검정과 KPI 산출에서 고장수리 270건만 골라내는 장면이 나오는데, 그 근거가 여기입니다. 다음은 장비별 건수입니다.

**[2-16] 장비별 정비 건수**

In [ ]:
print("장비 대수:", df["장비ID"].nunique())
df["장비ID"].value_counts().head()

장비는 모두 15대이고, 건수가 가장 많은 장비는 51건, 상위권 장비들이 30건대에 몰려 있습니다. 여기서 성급하게 "EX-205가 가장 문제"라고 말하면 안 됩니다. 앞에서 이야기한 노출 문제 때문입니다. 이 건수에는 정기점검이 섞여 있고, 장비마다 굴러다닌 기간과 현장 조건도 다릅니다. 지금 확인한 것은 순위가 아니라 "장비마다 건수가 고르지 않다"는 사실뿐입니다. 지금 보고 있는 숫자는 아직 전처리를 거치기 전의 숫자이기 때문입니다. 이 안에는 뒤에서 확인할 중복과 표기 변형이 섞여 있고, 정기점검과 고장수리도 갈라져 있지 않습니다. 구조 확인 단계의 목적은 결론을 내는 것이 아니라 데이터의 지형을 파악하고 다음 단계에서 확인할 목록을 만드는 것입니다. "장비별 건수 차이가 크다, 나중에 노출을 반영해 다시 본다"처럼 관찰과 숙제를 짝으로 적어 두면, 이 단계의 기록이 그대로 분석 계획이 됩니다.

### 표기 불일치 컬럼 탐지

**[2-18] 정비기사 표기 세기**

수기로 입력된 데이터의 표기 불일치 현상을 '정비기사' 열에서 확인해 보겠습니다.

In [ ]:
df["정비기사"].value_counts()

일곱 명이 일하는 팀인데 목록에는 열 개의 이름이 있습니다. 아래쪽 세 개를 보기 바랍니다. 김반장 7건, 박 기사님 5건, 최주임 4건입니다. 각각 김 반장, 박 기사, 최 주임과 같은 사람인데 띄어쓰기와 호칭이 달라 따로 집계되었습니다. 이대로 기사별 통계를 내면 어떻게 될까요. 김 반장은 23건 작업한 사람이 되고, 그와 별개로 김반장이라는 7건짜리 인물이 하나 더 생깁니다. 실제로는 30건입니다. 한 사람의 업무량이 둘로 쪼개져 양쪽 모두 실제보다 적게 보이는 것입니다. 이런 변형은 이름에서만 생기지 않습니다. 데이터 전체를 정리하는 과정에서 확인해 보면, 증상을 적는 칸에서도 같은 사건을 다르게 표현한 사례가 28건 있습니다. 사람이 자유롭게 적는 칸에서는 흔한 일입니다. 여기서 중요한 것은 이 문제를 기계가 알아서 처리하게 두면 안 된다는 점입니다. 비슷한 이름을 자동으로 묶으라고 하면, 실제로 다른 사람인 두 이름까지 합쳐 버릴 수 있습니다. 무엇을 무엇으로 바꿀지는 팀을 아는 사람이 정해야 합니다. 다음 실습에서 그 작평균 계산에서 빠집니다. 어느 쪽이 맞는지는 데이터가 아니라 업무가 정합니다. 결측을 다룰 때 먼저 구분해야 하는 것도 이것입니다. 「적어야 하는데 빠진 것」과 「적을 것이 없어서 빈 것」은 처리가 달라야 합니다. 정기점검 행의 원인부품이 비어 있는 것은 뒤쪽이고, 수리를 했는데 정비시간이 비어 있는 것은 앞쪽입니다. isna()는 각 칸이 이 NaN인지 아닌지를 확인하는 기능이고, sum()을 붙이면 컬럼별로 개수가 나옵니다.

## 5. 실습: 데이터 전처리

### 표기 통일: 인명 변형 표기 정규화

**[2-21] 정비기사 표기 통일**

정비기사 이름의 변형 표기를 정리하기 위해 다음과 같이 지시합니다. 표기 통일과 중복 정리정비기사 표기 변형을 정본으로 통일하고, 완전히 같은 중복 행을제거해줘. 같은 장비에서 30분 이내에 다시 입력된 유사 중복도 찾아줘. 이 지시에는 세 가지 작업이 들어 있습니다. 하나씩 나눠서 실행하겠습니다. 먼저 표기 통일입니다.

In [ ]:
mechanic_map = {"김반장": "김 반장", "박 기사님": "박 기사", "최주임": "최 주임"}
df["정비기사"] = df["정비기사"].replace(mechanic_map)
df["정비기사"].value_counts()

열 개였던 목록이 일곱 개로 줄었습니다. 김 반장은 23건에서 30건이 되었고, 최 주임은 29건에서 33건, 박 기사는 17건에서 22건이 되었습니다. 여기서 반드시 해야 할 검산이 있습니다. 통일 전후의 총합이 같아야 합니다. 통일 전 열 개 값의 합도 500이고, 통일 후 일곱 개 값의 합도 500입니다. 표기를 바꾸는 작업은 값을 옮겨 붙이는 것이지 지우는 것이 아니므로, 총합이 달라졌다면 어딘가에서 행이 사라졌다는 뜻입니다. 코드의 첫 줄을 다시 보기 바랍니다. 무엇을 무엇으로 바꿀지가 사람이 적은 목록으로 들어가 있습니다. 이 목록을 AI가 알아서 만들게 하고 싶은 유혹이 생기지만, 권하지 않습니다. 이름이 비슷하다는 이유로 실제로 다른 두 사람을 하나로 합쳐 버릴 수 있기 때문입니다. 우리 데이터에서는 김 반장과 김반장이 같은 사람이라는 것을 팀 사람은 알지만, 기계는 모릅니다. 바이브 코딩에서 사람이 개입해야 하는 자리가 바로 이런 곳입니다. 코드를 짜는 일은 맡기고, 무엇을 무엇으로 볼 것인지는 직접 정합니다.

### 완전 중복 행의 탐지와 제거

**[2-23] 완전 중복 행 찾기**

다음은 중복입니다. 모든 칸의 값이 완전히 같은 행부터 찾겠습니다.

In [ ]:
print("완전 중복 행:", df.duplicated().sum(), "건")
df[df.duplicated(keep=False)].sort_values("record_id").head(8)

중복은 4건인데 화면에는 8행이 나옵니다. 코드가 짝을 이루는 행을 모두 보여 주도록 되어 있기 때문입니다. 237번과 238번이 한 쌍, 244번과 245번이 한 쌍, 이런 식으로 네 쌍입니다. 주목할 점은 record_id까지 같다는 것입니다. MR-00238이 두 번 나옵니다. 정비 건마다 하나씩 붙는 고유 번호가 겹쳤다는 것은, 정비사가 같은 건을 두 번 입력했다기보다 파일을 합치거나 복사하는 과정에서 행이 복제되었을 가능성이 높다는 뜻입니다. 두 행의 모든 값이 한 글자도 다르지 않다는 점도 그 해석을 뒷받침합니다. 이런 판단이 왜 필요할까요. 원인을 알아야 재발을 막을 수 있기 때문입니다. 입력 습관의 문제라면 입력 화면을 고쳐야 하고, 파일 병합의 문제라면 병합 절차를 고쳐야 합니다. 지우는 것으로 끝나는 문제가 아닙니다.

**[2-25] 완전 중복 제거**

원인을 확인했으니 이제 제거하겠습니다.

In [ ]:
df = df.drop_duplicates().reset_index(drop=True)
print("제거 후 행수:", len(df))

500건에서 4건이 빠져 496건이 되었습니다. 빠진 개수와 앞에서 확인한 중복 건수가 일치합니다. 모든 열의 값이 일치하는 완전 중복 행을 탐색합니다. 관련해서 원본 보존의 원칙도 함께 적어 두겠습니다. 지금 우리가 지우고 바꾸는 것은 노트북으로 읽어 들인 복사본이지, 업로드한 CSV 원본이 아닙니다. 원본 파일은 손대지 않고 남겨 두고, 모든 처리는 코드로 수행해 노트북에 기록을 남기는 것이 안전한 작업 방식입니다. 이렇게 하면 처리 과정에 실수가 발견되어도 원본에서 처음부터 다시 시작할 수 있고, 노트북의 코드가 곧 "무엇을 어떻게 처리했는가"의 작업 일지가 됩니다. 엑셀에서 원본 파일을 직접 고치는 방식으로는 얻을 수 없는 이점입니다.

### 유사 중복: 기계 후보 추출과 사람 판정

**[2-27] 짧은 간격으로 연달아 기록된 행 찾기**

완전 중복은 쉬웠습니다. 값이 똑같으니 기계가 정확히 찾아냅니다. 문제는 같은 사건인데 값이 조금 다른 경우입니다. 데이터가 완벽히 일치하는 중복은 기계가 쉽게 찾지만, 동일한 사건임에도 입력 시각이 몇 분 다르거나 증상 표현이 미세하게 차이 나는 '유사 중복'은 찾아내기 어렵습니다. 이런 경우 동일한 장비에서 짧은 시간 내에 연달아 기록된 데이터를 의심 후보로 추출하는 접근 방식이 효과적입니다.

In [ ]:
df["일시"] = pd.to_datetime(df["고장일시"])
s = df.sort_values(["장비ID", "일시"]).reset_index(drop=True)
gap = s.groupby("장비ID")["일시"].diff()          # 같은 장비의 직전 기록과의 간격
suspect = s[gap <= pd.Timedelta(minutes=30)]
print("유사 중복 의심:", len(suspect), "건")
suspect[["record_id", "장비ID", "고장일시", "증상", "정비시간(분)", "정비기사"]]

코드의 첫 줄에서 문자로 읽혔던 고장일시를 날짜형으로 바꿨습니다. 이 변환을 하지 않으면 시간 간격을 계산할 수 없습니다. 그다음 장비별로 시간순 정렬하고, 직전 기록과의 간격이 30분 이하인 행을 뽑았습니다. 여기서 30분이라는 숫자에 주목하기 바랍니다. 이 값은 데이터가 알려 준 것이 아니라 사람이 정한 것입니다. 같은 장비에 30분 안에 두 건이 접수되는 일이 정상적으로는 드물다는 현장 감각에서 나온 기준입니다. 5분으로 잡으면 놓치는 것이 생기고, 세 시간으로 잡으면 관계없는 행까지 딸려 옵니다. 바이브 코딩에서 이런 기준값을 AI가 제안해 줄 수는 있습니다. 하지만 그 값이 우리 현장에 맞는지는 우리가 판단해야 합니다. 기준을 바꿔 가며 몇 건이 걸리는지 보고 정하는 것도 좋은 방법입니다.

**[2-29] 의심 행과 직전 행 나란히 보기**

4건이 걸렸습니다. 그런데 이 4건만 봐서는 판단할 수 없습니다. 무엇과 중복인지를 함께 봐야 하기 때문입니다. 의심 행과 그 직전 행을 짝으로 묶어 보겠습니다.

In [ ]:
# 의심 행과 그 직전 행을 나란히 놓고 눈으로 비교합니다.
pairs = sorted(set(suspect.index) | set(suspect.index - 1))
s.loc[pairs, ["record_id", "장비ID", "고장일시", "증상", "정비시간(분)", "정비기사"]]

이제 판정할 차례입니다. 네 쌍을 한 쌍씩 보겠습니다. 첫 번째 쌍(MR-00029와 MR-00494)은 10분 간격이고 증상이 "유압 작동 시 이상음"과 "유압 소리 이상"입니다. 같은 말을 다르게 적은 것으로 보입니다. 정비시간도 130분과 140분으로 비슷합니다. 세 번째 쌍(MR-00203과 MR-00496)은 "트랙 장력 이완"과 "트랙 헐거움"입니다. 역시 같은 상태를 두 표현으로 적었습니다. 네 번째 쌍(MR-00310과 MR-00495)은 "붐 실린더 누유"와 "붐실린더 누유"로, 띄어쓰기만 다릅니다. 이 세 쌍은 같은 사건을 다시 입력한 유사 중복으로 보입니다. 뒤에 입력된 쪽의 record_id가 MR-00494, MR-00495, MR-00496으로 번호가 몰려 있다는 점도 나중에 몰아서 추가 입력했다는 정황과 맞습니다. 문제는 두 번째 쌍입니다. MR-00052와 MR-00053은 20분 간격에 증상 표기가 "유압 작동 시 이상음"으로 완전히 같습니다. 앞의 세 쌍과 달리 표현이 갈리지 않았습니다. 재입력일 수도 있고, 실제로 20분 사이에 같은 증상으로 두 건이 접수된 것일 수도 있습니다. 정비시간도 185분과 205분으로 다릅니다. 데이터만 봐서는 판정할 수 없습니다. 여기서 확인해야 할 것은 원본입니다. 그날 정비일지에 작업이 한 번 적혀 있는지 두 번 적혀 있는지, 그날 근무했던 유 기사가 기억하는지를 확인해야 결론이 납니다. 기계는 후보를 찾아 줄 뿐이고, 판정은 원본을 확인한 사람이 합니다. 이것이 유사 중복 처리의 핵심입니다. 그래서 여기서는 기계적으로 지우지 않습니다. 확정되지 않은 건을 지우면 되돌릴 수 없지만, 남겨 두면 언제든 지울 수 있습니다. 제거 대상으로 표시만 해 두고 원본 확인을 요청하는 것이 안전한 순서입니다. 네 건이 전체 496건에서 차지하는 비중은 1% 미만이라 오늘의 통계를 크게 흔들지 않습니다. 그대로 두고 진행하되, 이 데이터로 보고서를 쓸 때 "유사 중복 후보 4건은 원본 확인 대기 중"이라는 사실을 함께 적으면 됩니다. 처리하지 않은 것과 처리하지 못한 것을 밝혀 두는 것도 분석의 일부입니다.

### 이상치 검토 원칙

**[2-32] 정비시간 기초 통계**

다음은 값이 유난히 큰 행입니다. 먼저 정비시간이 전체적으로 어떻게 퍼져 있는지 보겠습니다. 정비시간 분포와 이상치 확인정비시간의 기초 통계와 박스플롯을 보여 주고, 값이 가장 큰 기록몇 건을 조치방법과 함께 확인해 줘.

In [ ]:
df["정비시간(분)"].describe()

읽는 방법을 짚겠습니다. count가 476인 것은 496행 중 정비시간이 비어 있는 20건을 뺀 개수입니다. mean은 평균 132.9분, 50%는 중앙값 110분입니다. min은 가장 짧은 25분, max는 가장 긴 870분입니다. 여기서 눈여겨볼 것은 평균이 중앙값보다 23분이나 크다는 점입니다. 절반의 정비는 110분 안에 끝나는데 평균은 133분이라는 뜻입니다. 이런 차이는 한쪽에 아주 큰 값이 몇 개 있을 때 생깁니다. 실제로 최댓값이 870분, 열네 시간이 넘습니다. 숫자로 본 것을 그림으로 확인해 보겠습니다.

**[2-34] 정비시간 박스플롯**

In [ ]:
fig, ax = plt.subplots(figsize=(7, 2.4))
try:
    ax.boxplot(df["정비시간(분)"].dropna(), orientation="horizontal")
except TypeError:                       # 옛 matplotlib
    ax.boxplot(df["정비시간(분)"].dropna(), vert=False)
ax.set_xlabel("정비시간(분)")
ax.set_title("정비시간 분포 — 오른쪽 끝의 점이 이상치 후보")
plt.tight_layout(); plt.show()

실행하면 위와 같은 그래프가 나타납니다. 가운데 상자가 데이터의 절반이 모여 있는 구간이고, 상자에서 멀리 떨어져 오른쪽에 찍힌 점들이 이상치 후보입니다. 그림 하나로 앞의 숫자 여덟 개가 말하던 것을 한눈에 볼 수 있습니다. 이 그림은 뒤에서 검정 방법을 고를 때도 근거가 됩니다. 값이 한쪽으로 길게 늘어져 있으므로, 평균에 기대는 검정만 쓰면 안 되겠

**[2-35] 정비시간 상위 기록 확인**

이제 오른쪽 끝의 점들이 무엇인지 직접 확인하겠습니다. 이상치를 다룰 때 가장 중요한 단계입니다.

In [ ]:
df.sort_values("정비시간(분)", ascending=False)[
    ["record_id", "장비ID", "증상", "조치방법", "정비시간(분)"]].head(6)

600분을 넘는 건이 네 건입니다. 870분, 780분, 720분, 645분입니다. 조치방법 칸을 보기 바랍니다. 유압펌프 탈거·교체, 실린더 헤드 탈거와 개스킷 교체, 주행모터 탈거·교체, 하부주행체 일괄 교체입니다. 굴착기를 만져 본 사람이라면 바로 알 수 있습니다. 전부 하루를 통째로 쓰는 작업입니다. 유압펌프 하나 갈아 끼우는 데 열네 시간이 걸렸다는 기록은 이상한 값이 아니라 사실에 가깝습니다. 여기가 데이터를 아는 사람과 통계만 아는 사람이 갈리는 자리입니다. 통계 규칙만 적용하면 이 네 건은 "상자에서 멀리 떨어진 값" 이므로 제거 대상이 됩니다. 실제로 이상치를 자동으로 걸러 내는 코드는 흔합니다. 그런데 이 네 건을 지우면, 대형 수리라는 현장의 실제 사건이 데이터에서 사라집니다. 그러고 나면 "우리 정비는 평균 두 시간이면 끝난다"는 왜곡된 결론이 남습니다. 이상치는 지우는 것이 아니라 검토하는 것입니다. 그래서 여기서는 네 건을 모두 유지합니다. 실존 가능한 값이고, 오히려 중요한 사건이기 때문입니다. 다만 이 값들이 평균을 밀어 올린다는 사실은 그대로 남습니다. 그래서 뒤의 검정에서 순위만 사용하는 비모수 검정을 함께 돌리게 됩니다. 값을 지우는 대신 값에 덜 흔들리는 방법을 쓰는 것입니다. 데이터 중 유난히 큰 값들을 처리하기 위해 먼저 전체적인 분포를 확인합니다.

### 결측 복원: 보조 데이터 기반 대체

**[2-38] 가동 기록 파일 읽기**

가동 기록으로 결측 복원정비시간이 빈 행에 대해, 같은 장비 같은 날짜의 가동기록다운타임을 찾아줘. 그날 정비가 1건뿐이면 다운타임 값으로 채워줘. 먼저 가동 기록을 읽겠습니다.

In [ ]:
kpi = pd.read_csv("kpi/operation_2024_2025.csv")
print(kpi.shape)
kpi.head(3)

9,812행입니다. 장비 15대의 2년 치 일일 기록이니 이 정도 규모가 됩니다. 컬럼은 여덟 개인데, 지금 필요한 것은 장비ID, 일자, 다운타임(분) 세 개입니다. 나머지 컬럼은 KPI 실습에서 쓰겠습니다. 여기서 단위를 확인하고 넘어가기 바랍니다. 계획가동시간과 실제가동시간은 시간 단위이고, 다운타임만 분 단위입니다. 같은 표 안에서 단위가 섞여 있는 것은 현장 데이터에서 드물지 않은 일이며, 이 차이를 놓치면 뒤의 계산이 60배 어긋납니다.

**[2-40] 같은 날 다운타임으로 결측 채우기**

이제 복원하겠습니다.

In [ ]:
df["일자"] = df["일시"].dt.strftime("%Y-%m-%d")
day_cnt = df.groupby(["장비ID", "일자"])["record_id"].transform("count")
down = kpi.set_index(["장비ID", "일자"])["다운타임(분)"]
key = list(zip(df["장비ID"], df["일자"]))
miss = df["정비시간(분)"].isna()
single = miss & (day_cnt == 1)                     # 그날 정비가 1건뿐인 결측
df.loc[single, "정비시간(분)"] = [down[k] for k in df.index[single].map(lambda i: key[i])]
print(f"결측 {miss.sum()}건 중 {single.sum()}건 복원, 남은 결측 {df['정비시간(분)'].isna().sum()}건")

20건 중 19건이 복원되었습니다. 코드에서 눈여겨볼 곳은 single 줄입니다. 결측이면서 그날 그 장비의 정비가 1건뿐인 경우에만 채우도록 조건이 걸려 있습니다. 남은 1건은 그날 같은 장비에 정비가 두 건 있었던 경우입니다. 그날 다운타임이 300분이라 해도 두 건이 각각 몇 분씩이었는지는 알 수 없습니다. 물론 다른 한 건의 정비시간이 기록되어 있으니 빼면 구할 수는 있습니다. 그러나 그 계산은 "그날 다운타임이 두 정비의 합과 정확히 같다"는 전제 위에서만 성립합니다. 이동 시간이나 부품 대기 시간이 섞여 있으면 틀립니다. 이런 건은 원본 확인이 원칙입니다. 정비일지를 열어 보면 몇 분이었는지 적혀 있을 수 있고, 없다면 그대로 결측으로 남겨 두면 됩니다. 496건 중 1건은 통계에 영향을 주지 않습니다. 결측 처리의 정석은 삭제하거나 평균으로 대체하는 것이 아니라, 근거가 되는 다른 데이터를 찾는 것입니다. 이 방법이 가능했던 이유를 다시 짚어 두겠습니다. 회사에 가동 기록이라는 다른 데이터가 있었고, 두 데이터를 장비ID와 날짜로 이을 수 있었기 때문입니다. 어제 인벤토리를 만들면서 "무엇이 어디에 있는지" 목록을 만들어 둔 일이 여기서 값을 합니다. 어떤 데이터가 있는지 알아야 무엇으로 무엇을 채울 수 있는지도 보입니다. 전처리가 끝났습니다. 496행의 정제된 정비 이력이 준비되었고, 각 처리마다 왜 그렇게 했는지의 근거가 남았습니다. 이제 이 데이터로 질문에 답할 차례입니다.

## 6. 실습: 통계적 유의성 검정

### 계절성 검증: 카이제곱 검정

**[2-43] 겨울 ·저온성 증상 교차표**

겨울철 기온 하강에 따라 시동 관련 고장이 실제로 증가하는지 데이터로 확인해 보겠습니다. 겨울과 저온성 증상 교차표고장수리 건에서 겨울(11~2월) 여부와 저온성 증상 여부로교차표를 만들고 카이제곱 검정을 해줘.

In [ ]:
from scipy import stats
bd = df[df["구분"] == "고장수리"].copy()
bd["월"] = bd["일시"].dt.month
bd["겨울"] = bd["월"].isin([11, 12, 1, 2])
winter_kw = "시동|예열|동작 지연|반응 둔|수분|방전"
bd["저온성증상"] = bd["증상"].str.contains(winter_kw)
tbl = pd.crosstab(bd["겨울"], bd["저온성증상"])
tbl

코드를 세 부분으로 나눠 보겠습니다. 먼저 고장수리만 골라냈습니다. 정기점검은 계절과 무관하게 시간이 되면 하는 작업이므로 섞으면 안 됩니다. 다음으로 월을 뽑아 11·12·1·2월이면 겨울로 표시했습니다. 마지막으로 증상 문구에 시동·예열·동작 지연·반응 둔화·수분·방전 같은 표현이 들어 있으면 저온성 증상으로 표시했습니다. 교차표를 읽어 보겠습니다. 가로가 저온성 증상 여부, 세로가 겨울 여부입니다. 겨울이 아닌 기간에는 저온성 증상이 0건입니다. 176건 모두 다른 증상이었습니다. 반면 겨울에는 92건 중 41건이 저온성 증상입니다. 절반에 가깝습니다. 0이라는 숫자가 눈에 띕니다. 겨울이 아닌 기간에 시동 관련 문제가 단 한 건도 없었다는 뜻입니다. 이 정도로 깨끗하게 갈리면 검정을 하기 전에도 짐작이 갑니다. 그래도 확인은 해야 합니다.

**[2-45] 카이제곱 검정**

In [ ]:
chi2, p, dof, _ = stats.chi2_contingency(tbl)
print(f"카이제곱 = {chi2:.1f}, p = {p:.3e}")

p값이 3.579e-21로 나왔습니다. 지수 표기가 낯설 수 있는데, e-21은 소수점 아래 스물한 자리라는 뜻입 니다. 0.000000000000000000003579 정도의 값입니다. 기준인 0.05와 비교하면 압도적으로 작습니다. 계절과 저온성 증상이 서로 무관한데 우연히 이런 표가 나올 확률은 사실상 0이라는 뜻입니다. 겨울에 시동 관련 문제가 몰리는 것은 우연이 아닙니다. 이 결론을 현장 언어로 옮기면 이렇게 됩니다. 11월이 오기 전에 배터리 상태, 예열 장치, 연료 계통의 수분을 집중적으로 점검하면 겨울철 고장을 줄일 수 있습니다. 김 반장이 매년 11월마다 하던 일이 데이터로도 확인된 셈입니다. 한 가지 덧붙이면, 이 결과는 우리 데이터에서의 결과입니다. 장비 구성과 보관 환경이 다른 조직에서는 다르게 나올 수 있습니다. 검정은 우리 데이터가 무엇을 말하는지 알려 줄 뿐, 모든 현장에 적용되는 법칙을 만들어 주지는 않습니다.

### 기종별 정비시간 차이 검증: ANOVA와 비모수 검정

**[2-48] 장비ID로 기종 매핑하고 기종별 요약**

두 번째 질문입니다. 장비ID만으로는 기종을 알 수 없으므로 먼저 연결해 주어야 합니다. 기종별 정비시간 비교장비ID 앞 번호로 기종을 매핑하고, 기종별 고장수리 정비시간의평균·중앙값을 비교한 다음 ANOVA와 Kruskal-Wallis 검정을 해줘.

In [ ]:
eq_model = {
    "EX-101": "밥캣 E32",       "EX-102": "캐터필러 304 CR", "EX-103": "구보타 U27-4",
    "EX-104": "리파 R10-5",     "EX-201": "밥캣 E32",       "EX-202": "캐터필러 304 CR",
    "EX-203": "구보타 U27-4",   "EX-204": "밥캣 E32",       "EX-205": "리파 R10-5",
    "EX-206": "구보타 U27-4",   "EX-301": "캐터필러 304 CR", "EX-302": "밥캣 E32",
    "EX-303": "구보타 U27-4",   "EX-304": "리파 R10-5",     "EX-305": "밥캣 E32",
}
bd["기종"] = bd["장비ID"].map(eq_model)
bd.groupby("기종")["정비시간(분)"].agg(평균="mean", 중앙값="median", 건수="count").round(1)

매핑 사전은 사람이 만들어 넣어야 하는 정보입니다. 장비 관리대장에 있는 내용이지 데이터에 들어 있는 내용이 아니기 때문입니다. 이런 표를 만들어 두면 이후 분석에서 계속 재사용할 수 있습니다. 결과를 보겠습니다. 리파 R10-5가 평균 194.8분으로 가장 길고, 캐터필러 304 CR이 115.2분으로 가장 짧습니다. 차이가 80분 가까이 납니다. 평균과 중앙값을 함께 보면 더 많은 것이 보입니다. 구보타는 평균 124.5분인데 중앙값이 90분입니다. 34분 차이입니다. 절반의 정비는 90분 안에 끝나는데 평균이 그보다 훨씬 높다는 것은 몇 건의 긴 수리가 평균을 끌어올렸다는 뜻입니다. 반면 리파는 평균 194.8분, 중앙값 185분으로 차이가 10분밖에 나지 않습니다. 몇 건이 튀어서 평균이 높은 것이 아니라 전반적으로 오래 걸린다는 신호입니다. 건수도 봐야 합니다. 캐터필러는 45건뿐입니다. 앞에서 이야기한 소표본 변동이 가장 크게 작용할 집단입니다.

**[2-50] ANOVA와 Kruskal-Wallis 검정**

이제 이 차이가 우연인지 확인하겠습니다.

In [ ]:
groups = [g.dropna().values for _, g in bd.groupby("기종")["정비시간(분)"]]
f, p_a = stats.f_oneway(*groups)
h, p_k = stats.kruskal(*groups)
print(f"ANOVA          F = {f:.2f}, p = {p_a:.3e}")
print(f"Kruskal-Wallis H = {h:.2f}, p = {p_k:.3e}")

두 검정 모두 p값이 0.05보다 작습니다. 기종 간 정비시간 차이는 우연으로 보기 어렵습니다. 왜 두 가지를 돌렸을까요. 앞 실습에서 확인한 이상치 때문입니다. 870분짜리 수리가 섞여 있는 데이터에서 평균에 기 대 는 ANOVA만 쓰면, 그 몇 건이 결과를 좌우했을 가능성을 배제할 수 없습니다. Kruskal-Wallis는 값 자체가 아니라 순위를 쓰므로 큰 값 몇 개에 덜 흔들립니다. 두 방법이 같은 방향을 가리켰다는 점이 중요합니다. 접근 방식이 다른 두 검정이 같은 결론에 도달했다면 그 결론은 훨씬 단단합니다. 만약 ANOVA는 유의한데 Kruskal-Wallis는 유의하지 않게 나왔다면, 그때는 이상치가 결과를 만들어 낸 것은 아닌지 의심해야 합니다. 장비ID에 기종 정보를 매핑하여 기종별 고장수리 정비시간의 기초 통계량을 확인합니다.

### 특정 기종 대상 단측 검정

**[2-52] 리파 R10-5 단측 Mann-Whitney 검정**

리파 기종의 정비 시간이 가장 길다는 가설을 검정으로 확인합니다. 이번에는 "기종별로 정비 시간이 다른가"가 아니라 "리파가 나머지보다 긴가"라는 방향성을 가진 질문에 답하기 위해 단측 검정(One-tailed test)을 사용합니다. 단측 검정은 단순히 차이를 보는 것보다 더 명확한 결과를 제공하지만, 데이터를 보기 전 미리 방향을 정해야 한다는 규칙이 있습니다. 여기서는 "리파가 유독 오래 걸린다"는 현장의 사전 직관을 검증하는 것이므로 단측 검정 사용이 적절합니다. 단측 검정을 쓸 때는 지켜야 할 규율이 하나 있습니다. 방향은 데이터를 보고 나서가 아니라 데이터 밖의 근거로 미리 정해야 한다는 것입니다. 결과 표를 보고 큰 쪽을 골라 "크다"로 검정하면, 우연히 커 보이는 쪽에 유리한 판정을 내리는 셈이 되기 때문입니다. 우리의 방향에는 데이터 밖의 근거가 있습니다. 김 반장의 감은 표를 보기 전부터 리파를 가리키고 있었고, 우리는 그 주장을 검증하러 온 것입니다. 질문이 먼저 있었고 데이터가 나중에 온, 단측 검정의 정당한 사용입니다.

In [ ]:
r10 = bd.loc[bd["기종"] == "리파 R10-5", "정비시간(분)"].dropna()
rest = bd.loc[bd["기종"] != "리파 R10-5", "정비시간(분)"].dropna()
u, p_mw = stats.mannwhitneyu(r10, rest, alternative="greater")
print(f"리파 중앙값 {r10.median():.0f}분 vs 나머지 {rest.median():.0f}분,  단측 p = {p_mw:.3e}")

리파의 중앙값은 185분, 나머지 기종을 합친 중앙값은 105분입니다. 80분 차이입니다. p값은 0.05보다 훨씬 작습니다. 이제 우리는 말할 수 있습니다. 리파 R10-5는 다른 기종보다 정비시간이 깁니다. 중앙값 기준으로 80분, 비율로는 1.8배 가까이 됩니다. 이것은 인상이 아니라 확인된 사실입니다. 김 반장의 감이 처음으로 수치가 된 순간입니다. 회의에서 "저 녀석 유난히 손이 많이 가"라고 하던 말은, 이제 "리파 R10-5의 고장수리 정비시간 중앙값이 다른 기종의 1.8배이며 통계적으로 유의하다"는 문장이 되었습니다. 같은 내용이지만 후자는 회의록에 남고 결정의 근거가 됩니다. 다만 여기서 한 걸음 더 나가면 안 됩니다. 정비시간이 길다는 것과 고장이 잦다는 것은 다른 이야기입니다. 한 번 고장 날 때 오래 걸리는 것일 수도 있고, 자주 고장 나는데 매번 오래 걸리는 것일 수도 있습니다. 이 구분은 정비 이력만으로는 할 수 없습니다. 다음 실습에서 가동 기록을 함께 봐야 답이 나옵니다.

### 부트스트랩(Bootstrap) 신뢰구간 추정

**[2-55] 부트스트랩 신뢰구간**

부트스트랩 신뢰구간리파 R10-5의 정비시간 평균을 부트스트랩으로 재표집해95% 신뢰구간을 구해 줘.

In [ ]:
rng = np.random.default_rng(42)
boot = [rng.choice(r10, size=len(r10), replace=True).mean() for _ in range(10_000)]
lo, hi = np.percentile(boot, [2.5, 97.5])
print(f"리파 R10-5 평균 정비시간 {r10.mean():.0f}분 (95% CI {lo:.0f} ~ {hi:.0f}분, n={len(r10)})")

평균은 195분이고, 95% 신뢰구간은 167분에서 226분입니다. 폭이 60분 가까이 됩니다. 이 결과가 말하는 것은 이렇습니다. 우리가 가진 69건으로 추정한 리파의 평균 정비시간은 195분이지만, 표본이 조금 달랐다면 167분이 될 수도 226분이 될 수도 있었습니다. 195라는 숫자에 소수점까지 붙여 보고하는 것은 데이터가 뒷받침하지 않는 정밀함입니다. 스몰데이터의 결과는 숫자 하나가 아니라 구간으로 보고합니다. 실무에서 이 습관이 왜 중요한지 예를 들어 보겠습니다. "리파 평균 195분, 밥캣 평균 146분"만 보면 두 기종의 차이가 확실해 보입니다. 그런데 두 값 모두 60분 폭의 구간을 갖고 있다면 구간이 겹칠 수도 있습니다. 구간을 함께 보고하면 읽는 사람이 이 판단을 직접 할 수 있고, 하나의 숫자만 보고하면 그럴 기회 자체가 사라집니다.

## 7. 실습: 가동률·MTBF 산출

### 전체 가동률과 월별 추이

**단계 1. 전체 가동률 구하기**

**[2-57] 가동률 산식**

```
가동률(%) = 실제 가동 시간 합계 ÷ 계획 가동 시간 합계 × 100
```

**가동률과 기종별 KPI**

```
가동기록에서 전체 가동률(실제가동시간 합 ÷ 계획가동시간 합)과
월별 가동률 추이를 구하고, 기종별 가동률·MTBF·다운타임 표를 만들어줘.
```

**[2-59] 전체 가동률 계산**

In [ ]:
rate = kpi["실제가동시간"].sum() / kpi["계획가동시간"].sum() * 100
print(f"전체 가동률: {rate:.1f}%")

전체 가동률은 71.6%입니다. 계획했던 시간의 약 7할을 실제로 가동했다는 뜻입니다. 이 숫자가 좋은 것인지 나쁜 것인지는 이 값만으로 말할 수 없습니다. 임대업은 수요가 몰리는 시기와 비는 시기가 있고, 계획 가동 시간을 어떻게 잡느냐에 따라 값이 크게 달라집니다. 업종과 계약 형태가 다르면 비교 자체가 성립하지 않습니다. 가동률은 다른 회사와 견주는 숫자가 아니라 우리 회사의 흐름을 보는 숫자로 쓰는 편이 맞습니다.

**[2-61] 월별 가동률 추이 그래프**

그래서 흐름을 보겠습니다.

In [ ]:
kpi["월"] = kpi["일자"].str[:7]
monthly = (kpi.groupby("월")[["실제가동시간", "계획가동시간"]].sum()
             .eval("실제가동시간 / 계획가동시간 * 100"))
fig, ax = plt.subplots(figsize=(9, 3))
monthly.plot(ax=ax, marker="o")
ax.set_ylabel("가동률(%)"); ax.set_xlabel("")
ax.set_title("월별 가동률 추이 (2024-01 ~ 2025-11)")
plt.tight_layout(); plt.show()

### 기종별 가동률과 MTBF의 교차 해석

**[2-62] 기종별 가동률과 MTBF 계산**

이제 기종별로 나눠 보겠습니다. 여기서 새 지표가 하나 등장합니다. 기종별 KPI 산출을 위해 새로운 지표인 MTBF(Mean Time Between Failures, 평균 고장 간격)를 도입합니다. MTBF는 총 가동 시간을 고장 건수로 나눈 값으로, 장비가 고장 없이 얼마나 오래 작동했는지를 나타냅니다. 가동률이 '작업량'을 보여준다면, MTBF는 '고장 빈도'를 측정하는 척도가 됩니다. 가동률이 "얼마나 일했는가"를 말한다면 MTBF는 "얼마나 자주 멈췄는가"를 말합니다. 두 지표는 서로 다른 것을 봅니다. MTBF를 읽을 때 주의할 점이 하나 있습니다. 분모가 달력 시간이 아니라 실제 가동 시간이라는 점입니다. MTBF 90시간은 들여온 지 90시간 만에 고장 난다는 뜻이 아니라, 가동 시간이 90시간 쌓일 때마다 한 번꼴로 고장이 났다는 뜻입니다. 하루 여섯 시간씩 가동하는 장비라면 보름 남짓마다 한 번이라는 감각이 됩니다. 이렇게 가동 시간을 기준으로 삼기 때문에, 많이 굴린 장비와 적게 굴린 장비를 같은 잣대로 비교할 수 있습니다. 앞 장에서 본 노출 보정과 같은 발상이 지표의 정의 안에 이미 들어 있는 셈입니다.

In [ ]:
kpi["기종"] = kpi["장비ID"].map(eq_model)
by_model = kpi.groupby("기종").agg(
    실제=("실제가동시간", "sum"), 계획=("계획가동시간", "sum"),
    다운타임분=("다운타임(분)", "sum"), 고장=("고장건수", "sum"))
by_model["가동률(%)"] = (by_model["실제"] / by_model["계획"] * 100).round(1)
by_model["MTBF(h)"] = (by_model["실제"] / by_model["고장"]).round(1)
by_model["다운타임(h)"] = (by_model["다운타임분"] / 60).round(1)
by_model[["가동률(%)", "MTBF(h)", "다운타임(h)", "고장"]]

코드 중간에 다운타임을 60으로 나눈 줄이 보입니다. 앞에서 확인한 단위 차이를 여기서 맞춘 것입니다. 이 한 줄을 빠뜨리면 다운타임만 60배로 부풀어 오릅니다. 이제 표를 읽겠습니다. 먼저 가동률입니다. 68.6%, 69.5%, 74.8%, 71.5%. 네 기종이 모두 70% 안팎에 몰려 있습니다. 리파는 69.5%로 아래에서 두 번째인데, 가장 낮은 구보타와도 1%포인트 차이가 나지 않습니다. 가동률만 보면 "기종에 따른 차이가 별로 없다"는 결론이 나옵니다. 이번에는 MTBF를 보기 바랍니다. 캐터필러 199.6시간, 밥캣 177.9시간, 구보타 176.7시간, 그리고 리파 90.6시간입니다. 리파만 절반 수준입니다. 다른 기종이 180시간쯤 굴러야 한 번 고장 나는 데 비해, 리파는 90시간마다 한 번씩 멈춘다는 뜻입니다.

**[2-64] 기종별 가동률 ·MTBF 막대그래프**

이 대비를 그림으로 확인해 보겠습니다.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 3.2))
by_model["가동률(%)"].plot.bar(ax=axes[0], title="가동률(%)", rot=15)
by_model["MTBF(h)"].plot.bar(ax=axes[1], title="MTBF(시간)", rot=15)
plt.tight_layout(); plt.show()

실행하면 위와 같은 그래프가 나타납니다. 왼쪽은 가동률, 오른쪽은 MTBF입니다. 왼쪽 막대 네 개는 키가 비슷하고, 오른쪽에서는 리파 막대 하나만 눈에 띄게 짧습니다. 왜 이런 일이 생길까요. 가동률은 장비의 상태보다 임대 수요에 좌우되기 때문입니다. 계약이 잡히면 나가고 없으면 세워 둡니다. 고장이 잦아도 수리를 마치면 다시 나가므로, 웬만큼 고장이 나서는 가동률에 큰 흔적이 남지 않습니다. 반면 MTBF는 고장 건수를

### 이상 레코드 검토: 다운타임 초과 사례

**[2-66] 다운타임이 계획가동시간을 넘는 행 찾기**

KPI를 산출했으면 검산을 해야 합니다. 검산 중에 눈에 띄는 것이 하나 있습니다. 다운타임이 계획가동시간보다 큰 행입니다. 계획대로면 이런 행은 없어야 할 것 같습니다. 하루에 여섯 시간 굴리기로 한 장비가 여덟 시간을 멈춰 있었다는 말이 되기 때문입니다. 오류일까요. 몇 건인지부터 세어 보겠습니다. 다운타임 초과 행 확인다운타임이 계획가동시간보다 큰 행을 찾아서 비고별로 몇 건인지 세어 줘.

In [ ]:
over = kpi[kpi["다운타임(분)"] / 60 > kpi["계획가동시간"]]
print(len(over), "건")
over[["일자", "장비ID", "계획가동시간", "실제가동시간", "다운타임(분)", "비고"]].head(8)

20건입니다. 9,812행 중 20건이면 0.2%입니다. 이 정도면 그냥 지우고 싶어질 수 있습니다. 하지만 지우기 전에 열어 봐야 합니다. 행을 보겠습니다. 계획가동시간이 3.0, 3.5, 4.0, 8.0시간입니다. 실제가동시간은 전부 0입니다. 그날 장비는 한 시간도 일하지 못했습니다. 그리고 다운타임은 200분에서 720분까지입니다. 날짜를 보면 짐작이 갑니다. 2024년 6월 9일은 일요일이고, 계획가동시간이 3시간이나 4시간인 날은 주말이거나 동절기 단축 근무일입니다. 계획은 짧게 잡혀 있었는데 수리는 그 시간과 상관없이 걸렸던 것입니다. KPI 산출 후에는 반드시 검산 과정을 거쳐야 합니다. 검토 중 계획가동시간보다 다운타임이 더 길게 기록된 행들이 발견되었습니다. 계획된 시간보다 가동 중지 시간이 길다는 것은 논리적 오류처럼 보일 수 있으므로, 해당 사례가 몇 건인지 먼저 확인해 보겠습니다.

**[2-68] 초과 행의 비고 집계**

In [ ]:
over["비고"].value_counts()

20건 모두 이유가 적혀 있습니다. 18건은 고장 정지, 2건은 정기점검입니다. 정체불명의 행은 하나도 없습니다. 원인은 지표의 정의에 있습니다. 다운타임은 "수리에 쓴 시간"이지 "계획 시간 안에서 잃은 시간"이 아닙니다. 계획가동이 세 시간인 토요일에 다섯 시간짜리 수리를 하면 다운타임은 자연스럽게 계획을 넘습니다. 정의상 넘을 수 있는 값이니 오류가 아닙니다. KPI를 산출하기 전에 각 지표의 정의부터 확인해야 합니다. 만약 정의를 확인하지 않고 "말이 안 되는 행"이라며 20건을 지웠다면 어떻게 됐을까요. 지워진 20건은 전부 장비가 하루 종일 멈춰 있던 날, 그러니까 가장 큰 고장이 있었던 날입니다. 데이터에서 가장 중요한 사건만 골라 지우는 셈이 됩니다. 그 뒤에 산출한 MTBF와 다운타임 통계는 실제보다 훨씬 좋게 나왔을 것입니다. 이런 사고를 막는 실무 장치가 데이터 사전(Data Dictionary)입니다. 각 컬럼의 이름, 뜻, 단위, 값의 범위, 산출 규칙을 정리해 둔 문서를 말합니다. 다운타임은 수리에 쓴 시간이며 계획 가동 시간을 넘을 수 있다는 한 줄이 문서로 있었다면, 오늘 같은 확인 작업은 몇 분으로 끝났을 것입니다. 데이터를 만드는 부서와 읽는 부서가 다를수록 이 문서의 가치는 커집니다. 지금 당장 완전한 사전을 만들 수 없더라도, 오늘처럼 정의를 확인한 결과를 그때그때 적어 두는 것이 그 출발점이 됩니다. 이상치를 검토하고 유지했던 앞 실습과 같은 이야기입니다. 이상해 보이는 데이터를 만났을 때 첫 반응이 "지운다"가 되면 안 됩니다. 첫 반응은 "왜 이런 값이 나왔는지 확인한다"여야 합니다.

**[2-70] 관리자 보고 요약 생성 지시**

```
아래 KPI 산출 결과를 정비팀장에게 보고할 한 장 요약으로 정리해 주세요.
[산출 결과]
- 전체 가동률: (오늘 구한 값)
- 기종별 MTBF: (기종별 값)
- 확인한 예외: (건수와 확인 결과)
[요구 사항]
- 결론 한 줄로 시작할 것
- 근거로 드는 지표는 세 개 이내로 고를 것
- 확인이 필요한 항목과 다음 조치를 마지막에 적을 것
- 산출 결과에 없는 수치는 쓰지 말 것
- 원인을 단정하지 말고 확인이 필요한 사항으로 적을 것
```

## 8. 실습: 부품 이력 파일 통합

### 4개 파일의 컬럼 구조 비교

**[2-72] 부품 파일 4종 읽고 컬럼 비교**

먼저 네 파일이 어떻게 생겼는지 봅니다. 부품 파일 4종 통합부품 파일 4개의 컬럼 구조를 비교해줘. 표준 스키마(사용일자·장비ID·부품번호·부품명·수량·교체사유·출처)로 통합해줘.

In [ ]:
hq  = pd.read_excel("parts/parts_usage_hq.xlsx")
bs  = pd.read_excel("parts/parts_usage_busan.xlsx")
gj  = pd.read_csv("parts/parts_issue_gwangju.csv")
dl  = pd.read_csv("parts/parts_usage_dealer.csv")
for name, d in [("본사", hq), ("부산", bs), ("광주", gj), ("딜러", dl)]:
    print(f"{name:3s} {len(d):5d}행  {list(d.columns)}")

한눈에 봐도 제각각입니다. 같은 뜻인데 이름이 다릅니다. 본사의 사용일자는 부산에서 일자, 광주와 딜러에서 date이고, 부품번호는 품번·pn·part_no로 적혀 있습니다. 컬럼 개수도 본사·부산·딜러는 여섯 개인데 광주만 다섯 개라 무엇이 빠졌는지 확인해야 합니다. 행 수는 본사 2,000, 부산 1,300, 광주 1,000, 딜러 700으로 모두 5,000행입니다. 이름만 봐서는 알 수 없으니 실제 내용을 열어 보겠습니다. 부산부터입니다.

**[2-74] 부산 파일 앞 3행**

In [ ]:
bs.head(3)

두 가지가 눈에 띕니다. 날짜가 2024.01.02처럼 점으로 구분되어 있고, 개수 칸에 3개라는 문자가 섞여 있습니다. 첫 행은 3개인데 둘째 행은 1입니다. 사람이 손으로 입력한 흔적입니다. 이대로 두면 개수 칸으로 합계를 낼 수 없습니다. 문자가 섞인 열은 전체가 문자로 처리되기 때문입니다. 다음은 광주입니다.

**[2-76] 광주 파일 앞 3행**

In [ ]:
gj.head(3)

여기는 문제가 더 많습니다. 컬럼 이름이 영문 약어이고, 날짜가 01/02/2024 형식입니다. 이 형식은 월·일·연도 순서인데, 나라에 따라 일·월·연도로 읽기도 합니다. 어느 쪽으로 읽느냐에 따라 1월 2일이 될 수도 2월 1일이 될 수도 있습니다. 그리고 결정적으로 부품명이 없습니다. 품번만 있고 그 품번이 무슨 부품인지는 이 파일에 없습니다. 앞에서 컬럼이 하나 적었던 이유가 이것입니다. 본격적인 통합에 앞서 4개 파일의 구조를 확인하고, 표준 스키마(사용일자·장비ID·부품번호·부품명·수량·교체사유·출처)에 맞춰 통합 작업을 진행합니다.

### 주의 사례 1: 파일 소속과 장비 소속의 불일치

**[2-78] 본사 파일의 장비 소속 분포**

변환에 들어가기 전에 확인할 것이 하나 있습니다. 파일 이름과 내용이 일치하는지입니다. 본사 파일이니 본사 장비의 기록이 들어 있을 것 같습니다. 정말 그런지 세어 보겠습니다. 장비ID의 백의 자리로 소속을 구분할 수 있습니다.

In [ ]:
site = {"1": "본사", "2": "부산지점", "3": "광주지점"}
hq["장비ID"].str[3].map(site).value_counts()

데이터 변환 전, 파일 명칭과 실제 내용의 일치 여부를 점검해야 합니다. 본사 파일 내 장비들의 실제 소속을 장비ID의 특정 자릿수를 기준으로 분류하여 확인해 보겠습니다. 왜 이럴까요. 본사 창고가 전 거점의 정기 소모품을 일괄 구매해 배송하기 때문입니다. 엔진오일이나 그리스처럼 어느 장비에나 들어가는 품목은 본사가 한 번에 사서 지점으로 보냅니다. 그 출고 기록은 본사 시스템에 남고, 실제로 그 부품이 들어간 장비는 지점 장비입니다. 거점 조달이라고 부르는 이 구조 자체는 합리적입니다. 문제는 데이터를 읽는 사람이 그 구조를 모를 때 생깁니다. "본사 파일 2,000행"을 그대로 본사 장비의 부품 사용량으로 집계하면 실제의 네 배가 넘게 나옵니다. 이 확인이 가르쳐 주는 것은, 데이터에는 값만이 아니라 생성 맥락이 있다는 사실입니다. 이 파일이 어떤 업무 절차에서 만들어졌고 각 행이 어떤 사건을 기록한 것인지가 맥락입니다. 값은 파일 안에 있지만 맥락은 파일 밖, 그 업무를 아는 사람에게 있습니다. 본사 파일의 수수께끼도 부품 조달 절차를 아는 사람에게는 애초에 수수께끼가 아니었을 것입니다. 다른 부서의 데이터를 받을 때 파일만 받지 말고 "이 기록이 언제 어떻게 만들어지는지"를 한 번 물어보는 것이 좋은 습관인 이유가 여기에 있습니다. 파일 이름만 보고 내용을 단정하면 안 됩니다. 확인 방법은 파일을 열어 주요 컬럼의 분포를 한 번 세어 보는 것뿐입니다. 오늘 첫 실습에서 데이터 구조부터 확인했던 것과 같은 습관이며, 다른 조직에서 받은 파일이라면 더더욱 필요합니다.

### 표준 스키마 변환 절차

**[2-80] 본사 파일 표준 스키마 변환**

데이터 통합의 핵심은 목적지가 될 표준 규격을 먼저 정의하는 것입니다. 통일된 기준이 없으면 파일마다 수정 방향이 달라지기 때문입니다. 표준 컬럼은 총 7개(사용일자, 장비ID, 부품번호, 부품명, 수량, 교체사유, 출처)로 구성합니다. '출처'는 통합 후에도 데이터의 원천을 추적할 수 있도록 새로 추가한 항목입니다. 먼저 가공이 가장 수월한 본사 파일부터 변환하겠습니다.

In [ ]:
STD = ["사용일자", "장비ID", "부품번호", "부품명", "수량", "교체사유", "출처"]
h = hq.rename(columns={"수량(EA)": "수량"}).copy()
h["사용일자"] = pd.to_datetime(h["사용일자"])
h["출처"] = "본사"
h = h[STD]
h.head(2)

컬럼 이름 하나를 바꾸고, 날짜를 날짜형으로 바꾸고, 출처를 붙이고, 표준 순서대로 정렬했습니다. 마지막 줄에서 STD 목록의 순서대로 컬럼을 고른 덕분에 세 파일의 컬럼 순서가 같아집니다. 나중에 이어 붙일 때 이 순서가 맞아야 합니다. 출력의 첫 행이 EX-201, 둘째 행이 EX-302라는 점도 눈여겨보기 바랍니다. 방금 확인한 거점 조달 구조가 데이터에 그대로 보입니다. 날짜 형식과 문자 섞인 수량부산 파일에는 세 가지 처리가 필요합니다.

**[2-82] 부산 파일 표준 스키마 변환**

In [ ]:
b = bs.rename(columns={"일자": "사용일자", "장비": "장비ID", "품번": "부품번호",
                        "품명": "부품명", "개수": "수량", "사유": "교체사유"}).copy()
b["사용일자"] = pd.to_datetime(b["사용일자"], format="%Y.%m.%d")
b["수량"] = b["수량"].astype(str).str.replace("개", "").astype(int)   # '3개' → 3
b["출처"] = "부산지점"
b = b[STD]
b.head(2)

첫째, 컬럼 이름 여섯 개를 한꺼번에 바꿨습니다. 둘째, 날짜를 변환하면서 형식을 명시했습니다. 셋째, 수량에서 "개"를 떼어 내고 정수로 바꿨습니다. 첫 행의 3개가 3이 된 것을 확인할 수 있습니다. 날짜 변환에서 형식을 지정한 부분을 강조하고 싶습니다. 형식을 지정하지 않으면 도구가 알아서 해석하는데, 2024.01.02처럼 애매한 형태에서는 잘못 읽을 수 있습니다. 날짜가 하루씩 밀리거나 월과 일이 뒤바뀌면 그 뒤의 모든 집계가 어긋나는데, 그러고도 오류는 나지 않습니다. 날짜 형식은 데이터 통합에서 가장 자주 사고가 나는 자리이니, 변환한 뒤에는 앞 몇 행을 눈으로 확인하기 바랍니다. 없는 열은 마스터에서 가져오기광주 파일에는 부품명이 없었습니다. 없는 정보를 만들어 낼 수는 없지만, 다른 곳에서 가져올 수는 있습니다. 회사에는 부품 마스터가 있습니다. 품번과 부품명의 대응표입니다.

**[2-84] 광주 파일 표준 스키마 변환**

In [ ]:
master = pd.read_csv("parts/parts_master.csv")
g = gj.rename(columns={"date": "사용일자", "eq": "장비ID", "pn": "부품번호", "qty": "수량"}).copy()
g["사용일자"] = pd.to_datetime(g["사용일자"], format="%m/%d/%Y")
g = g.merge(master[["부품번호", "부품명"]], on="부품번호", how="left")   # 품번 → 부품명 복원
memo_map = {"정기교체": "정기교체", "정기 교체": "정기교체", "주기 도래": "정기교체",
            "고장": "고장수리", "고장수리": "고장수리", "수리건": "고장수리",
            "마모로 교체": "마모교체", "마모교체": "마모교체", "마모": "마모교체",
            "출고": "소모품 보충", "보충": "소모품 보충", "소모품": "소모품 보충"}
g["교체사유"] = g["memo"].map(memo_map)
g["출처"] = "광주지점"
g = g[STD]
g.head(2)

세 가지 처리가 들어 있습니다. 첫째, 날짜 형식을 월·일·연도로 명시했습니다. 01/02/2024를 1월 2일로 읽겠다고 지정한 것입니다. 이 판단의 근거는 데이터에 있습니다. 13 이상의 값이 앞자리에 나오는 행이 있는지 확인하면 어느 쪽이 월인지 알 수 있습니다. 둘째, 부품 마스터를 품번으로 이어 붙여 부품명을 복원했습니다. 참조 테이블에서 빠진 정보를 가져오는 이 방식은 현장에서 자주 씁니다. 다만 성립하려면 전제가 하나 필요합니다. 품번 체계가 거점마다 같아야 합니다. 광주가 자기만의 품번을 쓰고 있었다면 이 방법은 통하지 않습니다. 셋째, 사유 표기를 표준으로 맞췄습니다. "소모품", "보충", "출고"를 모두 "소모품 보충"으로 바꾸는 식입니다. 앞 실습에서 정비기사 이름을 통일했던 것과 같은 작업이며, 어떤 표현이 같은 뜻인지는 그 거점의 업무를 아는 사람만 압니다.

### 파일 통합 및 건수 검산

**[2-86] 세 파일 통합과 검산**

세 파일이 같은 모양이 되었으니 이어 붙이겠습니다.

In [ ]:
parts = pd.concat([h, b, g], ignore_index=True)
print("통합:", len(parts), "행 (본사 2,000 + 부산 1,300 + 광주 1,000)")
print("결측:", parts.isna().sum().sum(), "칸 | 마스터에 없는 품번:",
      (~parts["부품번호"].isin(master["부품번호"])).sum(), "건")
parts["교체사유"].value_counts()

검산 항목이 세 가지입니다. 행 수 4,300은 2,000 + 1,300 + 1,000과 정확히 같으니 변환 과정에서 사라진 행이 없습니다. 결측은 0칸인데, 광주의 부품명을 마스터에서 가져오는 작업이 모든 행에서 성공했다는 뜻이기도 합니다. 마스터에 없는 품번도 0건이라, 세 거점이 같은 품번 체계를 쓰고 있음이 확인되었습니다. 코드의 concat은 같은 스키마를 가진 표들을 세로로 이어 붙이는 기능입니다. 앞의 결측 복원에서 두 표를 키로 맞춰 가로로 이었던 것과 달리, 이번에는 같은 모양의 표를 위아래로 쌓는 단순한 연결입니다. 단순한 만큼 전제가 무겁습니다. 세 표의 컬럼 이름과 순서와 형식이 완전히 같아야 하며, 그 전제를 만드는 것이 지금까지의 변환 작업이었습니다. 통합 자체는 한 줄이지만, 그 한 줄이 성립하도록 만드는 데 이 실습의 대부분을 쓴 셈입니다. 교체사유도 표준 4종으로 정리되었습니다. 소모품 보충 2,351건, 마모교체 1,067건, 정기교체 547건, 고장수리 335건이고 네 값의 합도 4,300입니다. 구조를 통일한 세 데이터를 하나로 통합합니다.

### 주의 사례 2: 딜러 파일 제외 판단

**[2-88] 딜러 파일 앞 3행**

앞서 확인한 4개의 파일 중 딜러 파일(700행)을 제외한 3개만 통합했습니다. 해당 파일을 제외한 이유를 데이터 구성을 통해 확인하겠습니다.

In [ ]:
dl.head(3)

컬럼은 date, model, part_no, part_name, qty, reason 여섯 개입니다. 얼핏 다른 파일과 비슷해 보입니다. 그런데 장비ID가 없습니다. 대신 model이 있습니다. U27-4, R10-5, 304 CR은 장비 번호가 아니라 기종입니다. 어느 장비에 그 부품이 들어갔는지가 아니라, 어느 기종용 부품을 몇 개 샀는지의 기록입니다. 수량도 세 행 모두 4개씩으로, 같은 부품을 묶어 사는 구매의 모양입니다. 정리하면 이 파일은 성격이 다릅니다. 다른 세 파일이 "어느 장비에 어떤 부품을 썼다"는 출고 이력인 반면, 딜러 파일은 "어떤 부품을 몇 개 샀다"는 구매 이력입니다. 기록하는 사건 자체가 다릅니다. 이것을 무리하게 합치면 어떻게 될까요. 장비ID 칸이 비어 있는 700행이 생기고, 그 행들은 장비별 집계에서 전부 누락됩니다. 그런데 전체 합계에는 잡히므로 "부품 총 사용량 5,000건"이라는 숫자가 나옵니다. 실제 사용은 4,300건이고 나머지 700건은 아직 창고에 있을 수도 있는 물량인데 말입니다. 통합 전에 "이 파일이 같은 사건을 기록한 것인가"부터 판별해야 합니다. 이 판별이 오늘 마지막 실습의 진짜 목표였습니다. 양식을 맞추는 작업은 도구가 대신해 줍니다. 컬럼 이름을 바꾸고 날짜 형식을 통일하는 일은 자연어로 지시하면 몇 초 만에 코드가 나옵니다. 그러나 "이 파일을 합쳐도 되는가"라는 질문에는 도구가 답할 수 없습니다. 부품 조달 절차를 아는 사람만 답할 수 있습니다. 딜러 파일이 쓸모없다는 뜻은 아닙니다. 기종별 부품 구매량을 보거나 재고를 파악하는 데는 이 파일이 정확한 자료입니다. 다만 장비별 부품 이력과는 다른 표에 있어야 합니다.

**[2-90] 통합본 저장**

마지막으로 통합본을 파일로 남기겠습니다.

In [ ]:
parts.to_csv("parts_usage_merged.csv", index=False, encoding="utf-8-sig")
print("저장 완료: parts_usage_merged.csv")

인코딩을 지정한 이유는 한글이 깨지지 않게 하기 위해서입니다. 이 지정 없이 저장한 파일을 엑셀에서 열면 한글이 알 수 없는 문자로 보이는 일이 흔하니, 다른 사람이 열어 볼 파일이라면 반드시 넣어 두기 바랍니다. Colab에서 만든 파일은 세션이 끝나면 사라지므로 왼쪽 파일 탭에서 내려받아 두어야 합니다. 이 파일은 후속 PART의 지식베이스 구축과 자동화 실습에서 다시 쓰게 됩니다.

## 9. 실습: 고장 유형과 대응 우선순위

### 세 가지 기준의 상충

**단계 8. 파레토로 그려 보기**

---

## 마무리 — 오늘 코드를 파일로 받아 가기

아래 칸을 실행하면 **오늘 내가 실행한 코드 전부**가 `.py` 파일 하나로 내려받아집니다.
내가 고쳐 쓴 내용도 그대로 담기니, 회사에 돌아가 그대로 다시 돌려 볼 수 있습니다.

노트북 자체를 남기려면 「파일 → 드라이브에 사본 저장」도 함께 해 두세요.
막히는 곳은 학습사이트의 같은 절을 함께 보면 설명이 있습니다.

In [ ]:
# 오늘 실행한 코드를 파이썬 파일(.py) 한 장으로 내려받습니다.
# 이 칸은 실습을 다 끝낸 뒤 마지막에 한 번만 실행하세요.
import datetime
NL = chr(10)

runs = []
for n, cell in enumerate(In[1:], start=1):          # In = 지금까지 실행한 칸들
    if "files.download" in cell:                    # 이 칸 자신은 뺍니다
        continue
    keep = [ln for ln in cell.splitlines()          # 코랩 전용 명령(!apt-get 등)은 줄 단위로 뺍니다
            if not ln.lstrip().startswith(("!", "%"))]
    t = NL.join(keep).strip()
    if not t:                                       # 빈 칸도 뺍니다
        continue
    runs.append("# ─────────── 실행 " + str(n) + " ───────────" + NL + t + NL)

path = "DAY2_실습_내코드.py"
head = ("# DAY 2 실습 — 내가 실행한 코드 모음" + NL
        + f"# 저장 시각 {datetime.datetime.now():%Y-%m-%d %H:%M}" + NL
        + "# 코랩에서 실행한 순서 그대로입니다. 내가 고친 내용도 그대로 담깁니다." + NL + NL)
with open(path, "w", encoding="utf-8") as f:
    f.write(head + NL.join(runs))
print(path + " 저장 — 칸 " + str(len(runs)) + "개")

try:
    from google.colab import files
    files.download(path)                            # 내 PC로 내려받기
except ImportError:
    print("코랩이 아니면 왼쪽 폴더 아이콘에서 직접 내려받으세요.")